# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [3]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY Tai_lieu_ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [4]:
query_Tailieu = """
SELECT Tai_lieu_ID,
       Ma_tai_lieu,
       dbo.DecodeUTF8String(Nguoi_nhap_tin) AS Nguoi_nhap_tin,
       dbo.DecodeUTF8String(Nguoi_kiem_tra) AS Nguoi_kiem_tra,
       Nuoc_cung_cap_ID,
       Co_quan_cung_cap_ID,
       Ngay_giao_dich,
       Cap_mo_ta_thu_muc,
       Muc_do_mat,
       Vat_mang_tin_ID,
       Dang_tai_lieu_ID,
       Kieu_ban_ghi,
       Form_ID,
       Leader,
       Anh_bia,
      dbo.DecodeUTF8String(CallNumber) AS CallNumber
  FROM Tai_lieu
"""
df_tailieu = fetch_data_in_batches(query_Tailieu, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_tailieu)

C:\Users\admin\AppData\Local\Temp\ipykernel_30048\2025791915.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


       Tai_lieu_ID  Ma_tai_lieu      Nguoi_nhap_tin Nguoi_kiem_tra  \
0                7  SK020000011              DHSPKT           Điền   
1               12  SK020000015              DHSPKT           Điền   
2               26    SKV000004              DHSPKT           Điền   
3               30  SK020000030                luật           Điền   
4               31  SK020000032                  vi           Điền   
...            ...          ...                 ...            ...   
62282        67702  SK250067720  Nguyễn T. Hồng Nhi                  
62283        67703  SK250067721  Nguyễn T. Hồng Nhi                  
62284        67704  SK250067722  Nguyễn T. Hồng Nhi                  
62285        67705  SK250067723  Nguyễn T. Hồng Nhi                  
62286        67706  SK250067724  Nguyễn T. Hồng Nhi                  

      Nuoc_cung_cap_ID  Co_quan_cung_cap_ID      Ngay_giao_dich  \
0                 None                  NaN 2007-04-17 07:52:00   
1                 None   

C:\Users\admin\AppData\Local\Temp\ipykernel_30048\2025791915.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


### Tạo dataframe backup 

In [5]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_tailieu.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_tailieu_backup = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_tailieu:", len(df_tailieu)) # Kiểm tra số lượng dòng
print("Số dòng trong df_tailieu_backup:", len(df_tailieu_backup)) # Kiểm tra số lượng dòng

Số dòng trong df_tailieu: 62287
Số dòng trong df_tailieu_backup: 62287


### [Nếu cần] lấy lại data từ backup

In [6]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_tailieu_backup.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_tailieu = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_tailieu_backup:", len(df_tailieu_backup)) # Kiểm tra số lượng dòng
print("Số dòng trong df_tailieu:", len(df_tailieu)) # Kiểm tra số lượng dòng

Số dòng trong df_tailieu_backup: 62287
Số dòng trong df_tailieu: 62287


# Xử lý data

## Thêm 1 dòng giả định none

In [7]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'Tai_lieu_ID': [0],
    'Ma_tai_lieu': ['0'],
    'Nguoi_nhap_tin': ['(Không xác định)'],
    'Nguoi_kiem_tra': ['(Không xác định)'],
    'Nuoc_cung_cap_ID': [0],
    'Ngay_giao_dich': ['1024-01-01 00:00:00'],
    'Cap_mo_ta_thu_muc': ['0'],
    'Muc_do_mat': ['0'],
    'Vat_mang_tin_ID': [0],
    'Dang_tai_lieu_ID': [0],
    'Form_ID': [0],
    'Leader': ['(Invalid)'],
    'CallNumber': ['(Không xác định)']
})
# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_tailieu = pd.concat([df_tailieu, new_row], ignore_index=True) # Thêm vào dataframe
df_tailieu = df_tailieu.sort_values(by="Tai_lieu_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_tailieu)

       Tai_lieu_ID  Ma_tai_lieu      Nguoi_nhap_tin    Nguoi_kiem_tra  \
0                0            0    (Không xác định)  (Không xác định)   
1                7  SK020000011              DHSPKT              Điền   
2               12  SK020000015              DHSPKT              Điền   
3               26    SKV000004              DHSPKT              Điền   
4               30  SK020000030                luật              Điền   
...            ...          ...                 ...               ...   
62283        67702  SK250067720  Nguyễn T. Hồng Nhi                     
62284        67703  SK250067721  Nguyễn T. Hồng Nhi                     
62285        67704  SK250067722  Nguyễn T. Hồng Nhi                     
62286        67705  SK250067723  Nguyễn T. Hồng Nhi                     
62287        67706  SK250067724  Nguyễn T. Hồng Nhi                     

      Nuoc_cung_cap_ID  Co_quan_cung_cap_ID       Ngay_giao_dich  \
0                    0                  NaN  1024-01-01

## Xử lý data rỗng hoặc " "

In [8]:
df_tailieu = df_tailieu.replace('', None)
df_tailieu = df_tailieu.replace(np.nan, None)
print(df_tailieu)

       Tai_lieu_ID  Ma_tai_lieu      Nguoi_nhap_tin    Nguoi_kiem_tra  \
0                0            0    (Không xác định)  (Không xác định)   
1                7  SK020000011              DHSPKT              Điền   
2               12  SK020000015              DHSPKT              Điền   
3               26    SKV000004              DHSPKT              Điền   
4               30  SK020000030                luật              Điền   
...            ...          ...                 ...               ...   
62283        67702  SK250067720  Nguyễn T. Hồng Nhi              None   
62284        67703  SK250067721  Nguyễn T. Hồng Nhi              None   
62285        67704  SK250067722  Nguyễn T. Hồng Nhi              None   
62286        67705  SK250067723  Nguyễn T. Hồng Nhi              None   
62287        67706  SK250067724  Nguyễn T. Hồng Nhi              None   

      Nuoc_cung_cap_ID Co_quan_cung_cap_ID       Ngay_giao_dich  \
0                    0                None  1024-01-01 0

## Xử lý kiểu date

In [10]:
query_date = "SELECT Date_key FROM olap.DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_library)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_tailieu['Ngay_giao_dich'] = pd.to_datetime(df_tailieu['Ngay_giao_dich'], errors='coerce')
df_tailieu['Ngay_giao_dich'] = df_tailieu['Ngay_giao_dich'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
print(df_tailieu[['Ngay_giao_dich']])

C:\Users\admin\AppData\Local\Temp\ipykernel_30048\3478724221.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\admin\AppData\Local\Temp\ipykernel_30048\3478724221.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_tailieu['Ngay_giao_dich'] = pd.to_datetime(df_tailieu['Ngay_giao_dich'], errors='coerce')


       Ngay_giao_dich
0                   0
1            20070417
2            20070417
3            20070417
4            20070417
...               ...
62283        20250221
62284        20250221
62285        20250221
62286        20250221
62287        20250221

[62288 rows x 1 columns]


## Xử Lý Ma_Tai_lieu

In [11]:
df_tailieu['Ma_tai_lieu'] = df_tailieu['Ma_tai_lieu'].replace("", None)  # Thay giá trị chuỗi rỗng thành None
df_tailieu['Ma_tai_lieu'] = df_tailieu['Ma_tai_lieu'].fillna("(Invalid)")
print(df_tailieu[['Ma_tai_lieu']])

       Ma_tai_lieu
0                0
1      SK020000011
2      SK020000015
3        SKV000004
4      SK020000030
...            ...
62283  SK250067720
62284  SK250067721
62285  SK250067722
62286  SK250067723
62287  SK250067724

[62288 rows x 1 columns]


## Xử lý ID_mon

In [12]:
df_tailieu['ID_mon'] = 0
print(df_tailieu[['ID_mon']])

       ID_mon
0           0
1           0
2           0
3           0
4           0
...       ...
62283       0
62284       0
62285       0
62286       0
62287       0

[62288 rows x 1 columns]


## Xử lý ID_quoc_gia

In [14]:
query_Quocgia = "SELECT ID_quoc_gia FROM olap.DIM_Quoc_gia"
df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_library)
quocgia_ids = set(df_quocgia['ID_quoc_gia'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_quoc_gia của bảng DIM_Quoc_gia hay không ?
df_tailieu['Nuoc_cung_cap_ID'] = df_tailieu['Nuoc_cung_cap_ID'].apply(lambda x: x if pd.notna(x) and x in quocgia_ids else 0)
print(df_tailieu[['Nuoc_cung_cap_ID']])

       Nuoc_cung_cap_ID
0                     0
1                     0
2                     0
3                     0
4                     0
...                 ...
62283                 0
62284                 0
62285                 0
62286                 0
62287                 0

[62288 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_30048\3230730724.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quocgia = pd.read_sql(query_Quocgia, conn_dwh_library)


## Xử lý ID_vat_mang_tin

In [15]:
query_Vatmangtin = "SELECT ID_vat_mang_tin FROM olap.DIM_Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_library)
vatmangtin_ids = set(df_vatmangtin['ID_vat_mang_tin'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_vat_mang_tin của bảng DIM_Vat_mang_tin hay không ?
df_tailieu['Vat_mang_tin_ID'] = df_tailieu['Vat_mang_tin_ID'].apply(lambda x: x if pd.notna(x) and x in vatmangtin_ids else 0)
print(df_tailieu[['Vat_mang_tin_ID']])

       Vat_mang_tin_ID
0                    0
1                    3
2                    3
3                    3
4                    3
...                ...
62283                3
62284                3
62285                3
62286                3
62287                3

[62288 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_30048\1452113009.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_dwh_library)


## Xử lý ID_dang_tai_lieu

In [16]:
query_Dangtailieu = "SELECT ID_dang_tai_lieu FROM olap.DIM_Dang_tai_lieu"
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_library)
dangtailieu_ids = set(df_dangtailieu['ID_dang_tai_lieu'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Dang_tai_lieu_ID'] = df_tailieu['Dang_tai_lieu_ID'].apply(lambda x: x if pd.notna(x) and x in dangtailieu_ids else 0)
print(df_tailieu[['Dang_tai_lieu_ID']])

       Dang_tai_lieu_ID
0                     0
1                     1
2                     1
3                     1
4                     1
...                 ...
62283                 1
62284                 1
62285                 1
62286                 1
62287                 1

[62288 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_30048\1178039827.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_dwh_library)


## Xử lý ID_form

In [17]:
query_form = "SELECT ID_form FROM olap.DIM_Ten_form"
df_tenform = pd.read_sql(query_form, conn_dwh_library)
tenform_ids = set(df_tenform['ID_form'])
# chuyển date về dang int
# kiểm tra nhưng ngày đó có tồn tại trong ID_dang_tai_lieu của bảng DIM_Dang_tai_lieu hay không ?
df_tailieu['Form_ID'] = df_tailieu['Form_ID'].apply(lambda x: x if pd.notna(x) and x in tenform_ids else 0)
print(df_tailieu[['Form_ID']])

       Form_ID
0            0
1           83
2           83
3           83
4           83
...        ...
62283       37
62284       37
62285       37
62286       37
62287       37

[62288 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_30048\3707907086.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tenform = pd.read_sql(query_form, conn_dwh_library)


## Load data

### [Nếu cần] Clear bảng

In [18]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Tai_lieu"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [19]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO olap.DIM_Tai_lieu (
                    ID_tai_lieu, Ma_tai_lieu, 
                    Nguoi_nhap_tin, Nguoi_kiem_tra, 
                    ID_mon, ID_quoc_gia, ID_co_quan_cung_cap,
                    Ngay_giao_dich,
                    Cap_mo_ta_thu_muc, Muc_do_mat, 
                    ID_vat_mang_tin, ID_dang_tai_lieu,
                    Kieu_ban_ghi, ID_form, 
                    Leader, CallNumber
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['Tai_lieu_ID'], row['Ma_tai_lieu'],
        row['Nguoi_nhap_tin'], row['Nguoi_kiem_tra'],
        row['ID_mon'], row['Nuoc_cung_cap_ID'], row['Co_quan_cung_cap_ID'],
        row['Ngay_giao_dich'], 
        row['Cap_mo_ta_thu_muc'], row['Muc_do_mat'],
        row['Vat_mang_tin_ID'], row['Dang_tai_lieu_ID'], 
        row['Kieu_ban_ghi'], row['Form_ID'],
        row['Leader'], row['CallNumber'], 
    )
    for index, row in df_tailieu.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_library.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_library.close()